In [ ]:
from pathlib import Path
import sys


project_root = Path("..").resolve()

src_path = project_root / "src"

if not src_path.exists():
    raise FileNotFoundError(
        f"Could not find src directory: {src_path}"
    )

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from walinet.training_data.lcmodel_basis.parser import (
    load_lcmodel_basis,
)

basis = load_lcmodel_basis(
    "LCModelBasis/raw/7T.basis",
)

print(f"Metabolites : {basis.n_metabolites}")
print(f"Time points : {basis.n_points}")
print(f"Dwell time  : {basis.dwell_time:.9e} s")
print(f"Bandwidth   : {basis.bandwidth:.2f} Hz")
print(f"Hz / ppm    : {basis.hz_per_ppm:.3f}")

In [ ]:
from walinet.training_data.lcmodel_basis.hlsvd import (
    process_lcmodel_basis,
)

processed_basis = process_lcmodel_basis(
    basis,
    ppm_limits=(-0.2, 0.2),
    ppm_reference=4.65,
    n_singular_values=5,
    n_fit_points=8192,
)

print()
print("Finished.")
print(
    "Maximum reconstruction error:",
    (
        abs(
            processed_basis.original_fids
            - (
                processed_basis.clean_fids
                + processed_basis.reference_fids
            )
        )
    ).max(),
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_before_after_grid,
)

plot_basis_before_after_grid(
    basis,
    processed_basis,
    ppm_limits=(7.5, 0),
)

In [ ]:
from pathlib import Path

from walinet.training_data.lcmodel_basis.library import (
    build_or_extend_basis_library,
)


basis_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / "raw/7T.basis"
)

output_library_path = (
    project_root
    / "MetabModes/LCModelBasis"
    / "processed"
    / "walinet_7T_native_basis_v1.h5"
)


build_or_extend_basis_library(
    output_library_path,
    source_basis_path=basis_path,
    basis=basis,
    processed_basis=processed_basis,
    duplicate_policy="error",
    processing_repository_path=project_root,
)

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_basis_library_consistency,
)


plot_basis_library_consistency(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_v1.h5",
    ppm_limits=(7.5, 0.0),
    n_columns=4,
)

In [ ]:
from walinet.training_data.lcmodel_basis.acquisition import (
    prepare_basis_for_acquisition,
)


prepared_basis = prepare_basis_for_acquisition(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/processed/walinet_7T_native_basis_v1.h5",
    target_bandwidth=2778.0,
    target_n_timepoints=558,
)

In [ ]:
prepared_basis.fids.shape

In [ ]:
from walinet.training_data.lcmodel_basis.plotting import (
    plot_prepared_basis_grid,
)


plot_prepared_basis_grid(
    prepared_basis,
    ppm_limits=(7.5, 0),
    n_columns=4,
)